# Q1: Why do MLPs struggle on CIFAR-10, and do they learn structure before memorising?

**Notebook structure**

0. Configuration (set `ROLL_NUMBER`)
1. Data: stratified 45,000 / 5,000 split seeded by the roll number
2. Shared components (models, counters, training loop): introduced here, reused in Q1(a) and Q1(b)
3. **Q1(a)**: missing spatial inductive bias vs. capacity vs. optimisation
4. **Q1(b)**: memorisation under label noise, analysed from the training trajectory
5. Final test-set evaluation (done once, after every choice is fixed)
6. Interpretation of the evidence (generated from the measured numbers)
7. Reuse log

**Data protocol.** All selection (learning rate, early-stopping epoch, the choice of models) uses the 5,000-image
validation split only. The official CIFAR-10 test set is not loaded until Section 5, after every choice has been made.
Every final number quoted as a result in Sections 5 and 6 is a test-set number. Training and validation curves are shown where the question asks for training behaviour.

**Runtime.** The notebook is sized for a single GPU (for example a Colab T4: *Runtime → Change runtime type → GPU*).
The whole budget is 35 runs of 30 epochs for Q1(a) plus two 150-epoch runs for Q1(b), which should take on the order
of an hour on a T4. The convolutional runs dominate the cost. Each run is cached in `q1_outputs/runs/`, so if the session
disconnects you can re-run the notebook and it resumes where it stopped. On a CPU the full budget would take many hours.
To cut the cost, reduce `N_SEEDS` or `CFG_A["epochs"]`, and report the change.

In [ ]:
# ===================== 0. CONFIGURATION (edit ROLL_NUMBER) =====================
ROLL_NUMBER = "ENTER_YOUR_ROLL_NUMBER"   # e.g. "21CS10045" -> seed 2110045 (all digits, in order)

DATA_ROOT = "./data"          # torchvision downloads CIFAR-10 here
OUT_DIR   = "./q1_outputs"    # figures, tables, cached runs

N_TRAIN, N_VAL = 45_000, 5_000

# Q1(a): one training budget shared by every model
CFG_A = dict(epochs=30, batch_size=128, weight_decay=0.05, schedule="cosine")
LR_GRID = [3e-4, 1e-3, 3e-3]   # identical LR grid for every model; the choice is made on validation
N_SEEDS = 3                    # training runs per model at its selected LR
TRAIN_EVAL_SUBSET = 5_000      # fixed training subset re-evaluated in eval mode every epoch

# Q1(b): memorisation experiment
CFG_B = dict(epochs=150, batch_size=128, lr=1e-3, weight_decay=0.0, schedule="constant")
NOISE_RATE = 0.4               # fraction of training labels replaced by a wrong class
DENSE_EVAL_EPOCHS_B = 10       # evaluate 4x per epoch for the first 10 epochs, then once per epoch

In [ ]:
import os, math, time, json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from scipy import stats
from sklearn.model_selection import train_test_split
from IPython.display import display

warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")
pd.set_option("display.width", 200)

digits = "".join(ch for ch in str(ROLL_NUMBER) if ch.isdigit())
if not digits:
    raise ValueError("Set ROLL_NUMBER in the configuration cell first.")
SEED = int(digits)

def derive_seed(*tags):
    '''32-bit seed derived deterministically from the roll-number seed plus a tag,
    so every random choice in the notebook traces back to SEED.'''
    words = [SEED] + [int.from_bytes(str(t).encode(), "little") for t in tags]
    return int(np.random.SeedSequence(words).generate_state(1)[0])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True
os.makedirs(os.path.join(OUT_DIR, "runs"), exist_ok=True)

# Plot style: fixed-order colour-blind-checked categorical palette, recessive grid
PALETTE = ["#257de0", "#f47230", "#00b894", "#f2a600", "#ee7fac", "#008a17", "#6a4ce2", "#ec4647"]
GREY = "#7a7a7a"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.prop_cycle": plt.cycler(color=PALETTE), "axes.grid": True,
    "grid.color": "#e7e7e7", "grid.linewidth": 0.8, "axes.edgecolor": "#c2c2c2",
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "legend.frameon": False, "font.size": 10,
})
def savefig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, name))
    plt.show()

def pct(x):
    return f"{100 * x:.1f}%"

def pp(x):
    return f"{100 * x:+.1f} pp"

print(f"Roll number {ROLL_NUMBER!r} -> seed {SEED}")
print(f"device={DEVICE}  AMP={USE_AMP}  torch={torch.__version__}")
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. The full budget is sized for a GPU; on CPU expect many hours.")

## 1. Data and the common split

The 50,000 CIFAR-10 training images are split once into **45,000 train / 5,000 validation**, stratified by class
(4,500 / 500 per class). `sklearn.model_selection.train_test_split` performs the split, with the digits of the roll
number, in order, as `random_state`. Per-channel normalisation statistics are computed on the 45k training split only.
The split indices are saved to `q1_outputs/split_indices.npz` so later questions can reuse exactly the same split.

In [ ]:
train_full = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=True)
X_all = np.asarray(train_full.data)                       # (50000, 32, 32, 3) uint8
y_all = np.asarray(train_full.targets, dtype=np.int64)
CLASSES = list(train_full.classes)

# sklearn needs a 32-bit integer; a roll number with more than 9-10 digits is mapped through SeedSequence
split_state = SEED if SEED < 2**32 else derive_seed("split")
train_idx, val_idx = train_test_split(np.arange(len(y_all)), train_size=N_TRAIN, test_size=N_VAL,
                                      stratify=y_all, random_state=split_state)
np.savez(os.path.join(OUT_DIR, "split_indices.npz"), train_idx=train_idx, val_idx=val_idx, seed=SEED)

split_counts = pd.DataFrame({"train": np.bincount(y_all[train_idx], minlength=10),
                             "validation": np.bincount(y_all[val_idx], minlength=10)}, index=CLASSES)
assert len(np.intersect1d(train_idx, val_idx)) == 0
assert split_counts["train"].sum() == N_TRAIN and split_counts["validation"].sum() == N_VAL
print(f"random_state used for the split: {split_state}")
display(split_counts.T)

# Per-channel normalisation from the training split only
_x = torch.from_numpy(X_all[train_idx]).float().div_(255)
CH_MEAN, CH_STD = _x.mean(dim=(0, 1, 2)), _x.std(dim=(0, 1, 2))
del _x
print("channel mean", CH_MEAN.numpy().round(4), " std", CH_STD.numpy().round(4))

def to_tensor(x_u8):
    '''uint8 NHWC -> normalised float32 NCHW.'''
    x = torch.from_numpy(np.ascontiguousarray(x_u8)).float().div_(255)
    return ((x - CH_MEAN) / CH_STD).permute(0, 3, 1, 2).contiguous()

# The whole split lives on the GPU: no DataLoader overhead, and no augmentation is used anywhere in Q1
Xtr, ytr = to_tensor(X_all[train_idx]).to(DEVICE), torch.from_numpy(y_all[train_idx]).to(DEVICE)
Xva, yva = to_tensor(X_all[val_idx]).to(DEVICE), torch.from_numpy(y_all[val_idx]).to(DEVICE)
print("train", tuple(Xtr.shape), " validation", tuple(Xva.shape))

In [ ]:
# Fixed pixel permutation, used by the Q1(a) control conditions
PIXEL_PERM = torch.from_numpy(
    np.random.default_rng(derive_seed("pixel-permutation")).permutation(32 * 32)).to(DEVICE)

def identity(x):
    return x

def permute_pixels(x):
    '''Apply ONE fixed permutation to the 1024 pixel positions of every image.
    The 3 colour values of a pixel move together, so the information in the image and its colour
    statistics are unchanged; only the spatial arrangement is destroyed.'''
    n = x.shape[0]
    return x.reshape(n, 3, 32 * 32)[:, :, PIXEL_PERM].reshape(n, 3, 32, 32)

def to_display(x):
    return (x.permute(1, 2, 0).cpu() * CH_STD + CH_MEAN).clamp(0, 1).numpy()

fig, axes = plt.subplots(2, 6, figsize=(9, 3.4))
for j in range(6):
    i = int(np.where(y_all[train_idx] == j)[0][0])
    axes[0, j].imshow(to_display(Xtr[i]))
    axes[0, j].set_title(CLASSES[j], fontsize=9)
    axes[1, j].imshow(to_display(permute_pixels(Xtr[i:i + 1])[0]))
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
axes[0, 0].set_ylabel("original")
axes[1, 0].set_ylabel("permuted")
fig.suptitle("The same images after the fixed pixel permutation (Q1(a) control)")
savefig(fig, "fig_a0_permutation_examples.png")

## 2. Shared components

These models, counters and the training loop are **introduced here and reused unchanged in Q1(a) and Q1(b)**
(and can be reused by later questions).

* `MLP(width, depth=3)`: `Flatten → [Linear → BatchNorm1d → ReLU] × 3 → Linear`. Depth is fixed and only the width varies, so the capacity sweep changes one factor.
* `CNN()`: six 3×3 convolutions (BN + ReLU) in three stages with 2×2 max-pooling, then global average pooling and a linear head. It is a deliberately plain VGG-style network with no residual connections, dropout or other tricks, so that "convolution" is the main difference from the MLP.
* Both families use the same normalisation (BatchNorm), activation (ReLU), loss, optimiser (AdamW), LR schedule and mixed-precision setting.
* `train_model`: AdamW with a cosine or constant LR. Every epoch it logs the running training loss and accuracy, the **eval-mode** loss and accuracy on a fixed 5k training subset (comparable with validation), and validation loss and accuracy. It keeps the checkpoint with the best validation accuracy, so early stopping is selected on validation only.

In [ ]:
class MLP(nn.Module):
    '''Flatten -> [Linear -> BatchNorm1d -> ReLU] x depth -> Linear.'''
    def __init__(self, width, depth=3, in_dim=3 * 32 * 32, n_classes=10):
        super().__init__()
        layers, d = [nn.Flatten()], in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.BatchNorm1d(width), nn.ReLU(inplace=True)]
            d = width
        layers.append(nn.Linear(d, n_classes))
        self.net = nn.Sequential(*layers)
        self.spec = "MLP 3072 -> " + " -> ".join([str(width)] * depth) + " -> 10 (Linear+BN+ReLU per hidden layer)"

    def forward(self, x):
        return self.net(x)


def conv_bn_relu(cin, cout):
    return [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]


class CNN(nn.Module):
    '''Six 3x3 conv layers (BN+ReLU) in three stages with 2x2 max-pooling, global average pooling, linear head.'''
    def __init__(self, ch=(32, 64, 128, 128, 256, 256), n_classes=10):
        super().__init__()
        c1, c2, c3, c4, c5, c6 = ch
        self.features = nn.Sequential(
            *conv_bn_relu(3, c1), *conv_bn_relu(c1, c2), nn.MaxPool2d(2),     # 32x32 -> 16x16
            *conv_bn_relu(c2, c3), *conv_bn_relu(c3, c4), nn.MaxPool2d(2),    # 16x16 -> 8x8
            *conv_bn_relu(c4, c5), *conv_bn_relu(c5, c6), nn.MaxPool2d(2))    # 8x8   -> 4x4
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c6, n_classes))
        self.spec = (f"CNN conv3x3 [{c1},{c2}] pool | [{c3},{c4}] pool | [{c5},{c6}] pool | GAP | "
                     f"Linear {c6}->10 (BN+ReLU after each conv)")

    def forward(self, x):
        return self.head(self.features(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_macs(model, input_shape=(1, 3, 32, 32)):
    '''Multiply-accumulates of conv and linear layers for one image (BN, ReLU and pooling ignored).'''
    macs = 0
    def conv_hook(m, inp, out):
        nonlocal macs
        macs += out.numel() * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]
    def lin_hook(m, inp, out):
        nonlocal macs
        macs += out.numel() * m.in_features
    hooks = [m.register_forward_hook(conv_hook) for m in model.modules() if isinstance(m, nn.Conv2d)]
    hooks += [m.register_forward_hook(lin_hook) for m in model.modules() if isinstance(m, nn.Linear)]
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(input_shape, device=next(model.parameters()).device))
    model.train(was_training)
    for h in hooks:
        h.remove()
    return macs


def mlp_param_count(width, depth=3, in_dim=3072, n_classes=10):
    '''Closed form for MLP(width, depth): Linear weights+biases plus BatchNorm scale/shift.'''
    first = in_dim * width + width + 2 * width
    hidden = (depth - 1) * (width * width + width + 2 * width)
    return first + hidden + width * n_classes + n_classes


def width_matching(target, depth=3):
    '''MLP width whose parameter count is closest to `target`.'''
    return min(range(8, 8192), key=lambda w: abs(mlp_param_count(w, depth) - target))

In [ ]:
@torch.no_grad()
def predict_logits(model, X, input_fn=identity, bs=2000):
    model.eval()
    out = []
    for i in range(0, X.shape[0], bs):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(model(input_fn(X[i:i + bs])).float())
    return torch.cat(out)


def loss_acc(logits, y):
    return F.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item()


def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def df_to_lists(df):
    return {c: df[c].astype(float).tolist() for c in df.columns}


def train_model(model, X, y, Xv, yv, *, epochs, lr, weight_decay, batch_size, schedule, seed,
                input_fn=identity, train_eval_idx=None, step_callback=None, callback_steps=(),
                tag="", print_every=10):
    '''AdamW + (cosine | constant) LR schedule, mixed precision on GPU.
    Per epoch: running train loss/acc, eval-mode loss/acc on a fixed training subset, validation loss/acc.
    Keeps the checkpoint with the best validation accuracy (early-stopping selection uses validation only).'''
    model = model.to(DEVICE)
    n = X.shape[0]
    steps_per_epoch = math.ceil(n / batch_size)
    total_steps = epochs * steps_per_epoch
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    if schedule == "cosine":
        lr_lambda = lambda s: 0.5 * (1 + math.cos(math.pi * min(s, total_steps) / total_steps))
    elif schedule == "constant":
        lr_lambda = lambda s: 1.0
    else:
        raise ValueError(schedule)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    gen = torch.Generator().manual_seed(seed)              # data order depends only on the run seed
    callback_steps = set(callback_steps)
    rows, best, step, t0 = [], dict(val_acc=-1.0, epoch=0, state=None), 0, time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(n, generator=gen).to(DEVICE)
        loss_sum = torch.zeros((), device=DEVICE)
        n_correct = torch.zeros((), device=DEVICE)
        for i in range(0, n, batch_size):
            idx = order[i:i + batch_size]
            xb, yb = input_fn(X[idx]), y[idx]
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            loss_sum += loss.detach().float() * len(idx)
            n_correct += (logits.argmax(1) == yb).sum()
            step += 1
            if step_callback is not None and step in callback_steps:
                step_callback(model, step / steps_per_epoch)
                model.train()

        row = dict(epoch=epoch, lr=sched.get_last_lr()[0],
                   train_loss_running=loss_sum.item() / n, train_acc_running=n_correct.item() / n)
        row["val_loss"], row["val_acc"] = loss_acc(predict_logits(model, Xv, input_fn), yv)
        if train_eval_idx is not None:
            row["train_loss"], row["train_acc"] = loss_acc(
                predict_logits(model, X[train_eval_idx], input_fn), y[train_eval_idx])
        rows.append(row)
        if row["val_acc"] > best["val_acc"]:
            best = dict(val_acc=row["val_acc"], epoch=epoch, state=cpu_state(model))
        if print_every and (epoch == 1 or epoch % print_every == 0 or epoch == epochs):
            extra = f" train(eval) {row['train_acc']:.3f}" if "train_acc" in row else ""
            print(f"  [{tag}] ep {epoch:3d}/{epochs} loss {row['train_loss_running']:.3f} "
                  f"train(run) {row['train_acc_running']:.3f}{extra} val {row['val_acc']:.3f} "
                  f"({time.time() - t0:.0f}s)")

    return dict(history=pd.DataFrame(rows), best_epoch=best["epoch"], best_val_acc=best["val_acc"],
                best_state=best["state"], final_state=cpu_state(model), seconds=time.time() - t0,
                steps=step)

## 3. Q1(a): Is the MLP's problem a missing spatial inductive bias, or something else?

### Chosen reason (H1): no spatial inductive bias
An MLP's first layer connects every pixel to every hidden unit with an independent weight. It has no built-in notion
that neighbouring pixels are related (**locality**), and a feature it learns at one image position does not carry over to
another (**no weight sharing, no translation equivariance**). From 45k images it must learn every local pattern
separately at every position, so it can fit the training set with position-specific pixel combinations that do not
transfer to new images. H1 therefore predicts that the MLP **overfits rather than underfits**, and that a model which
builds in locality and weight sharing does much better *only when the input actually has spatial structure*.

### Competing explanations
* **H2, insufficient capacity:** the MLP is too small or weak to represent a good classifier.
* **H3, optimisation difficulty:** a good solution exists in the MLP's parameter space, but AdamW does not find it from raw pixels.

### Design: a 2 × 2 factorial at a matched budget, plus a capacity sweep

| | original images | fixed pixel permutation |
|---|---|---|
| **MLP-M** (≈ CNN's parameter count) | MLP-M | MLP-M-perm |
| **CNN** | CNN | CNN-perm |

The **pixel permutation** is the separating control. It applies one fixed shuffle of the 1024 pixel positions to every
train, validation and test image. It leaves unchanged the information in each image, the labels, the parameter count, the FLOPs, the
optimiser and the data budget, and removes only the spatial arrangement that convolution relies on. An MLP is
mathematically indifferent to it: a permutation of the inputs is absorbed by permuting the columns of the first weight
matrix, and i.i.d. initialisation is permutation-invariant. So MLP-M vs MLP-M-perm also measures run-to-run noise.

A **capacity sweep** (MLP-S/M/L/XL, widths 128 → 2048, about 0.4M → 14.7M parameters, depth fixed) tests H2 directly. The
**training-set fit** of every model (eval-mode accuracy and loss on a fixed 5k training subset, every epoch) tests H2 and H3.

### Predictions (stated before running)

| Measurement | H1 inductive bias | H2 capacity | H3 optimisation |
|---|---|---|---|
| MLP training accuracy | high, so a large train–validation gap | low (underfits) | low, training loss stalls |
| Widening the MLP ~34× | small validation gain | large validation gain | little change in fit |
| CNN at an equal parameter count | much better | similar to the MLP | maybe better (easier to optimise) |
| Permuting pixels: CNN | large drop, to MLP level or below | no change (capacity is unchanged) | no change |
| Permuting pixels: MLP | no change | no change | no change |

Only H1 predicts an **interaction**: the CNN's advantage exists on real images and disappears when the spatial structure
is removed, while parameters and compute stay fixed.

### What counts as a fair comparison here
1. **Same data and preprocessing.** Every model uses the identical 45k/5k split and per-channel normalisation from the training split, with **no data augmentation for any model**. Random crops and flips hard-code translation and mirror invariance, so they would inject the very spatial prior being tested, and they are not meaningful on permuted images.
2. **Matched parameters.** MLP-M's width is solved so that its parameter count is within ~0.2% of the CNN's.
3. **Matched training budget.** All models get the same epochs and batch size, and therefore the same number of gradient steps and example presentations. They also share the optimiser (AdamW), cosine schedule, weight decay, loss, normalisation layer (BatchNorm) and activation (ReLU).
4. **Equal tuning effort.** Every model gets the same 3-point learning-rate grid, and the LR and the early-stopping epoch are chosen on validation accuracy only. The permuted conditions are tuned separately too, so that a poorly chosen LR cannot explain a drop.
5. **Repeated runs.** Each model is retrained with `N_SEEDS` seeds at its selected LR, and differences are judged against the seed-to-seed spread. A given seed index shares its initialisation stream and batch order across models, which makes the paired comparisons less noisy.
6. **What is not matched, and why that is acceptable.** Compute per image is not matched. Weight sharing lets the CNN reuse each parameter at many positions, so it performs far more multiply-accumulates per parameter (reported in the table below). Matching FLOPs instead would force either a tiny CNN or a huge MLP. The permutation control removes this confound: CNN-perm has exactly the CNN's parameters **and** FLOPs, so a drop cannot be blamed on compute.

In [ ]:
CNN_PARAMS = count_params(CNN())
W_MATCH = width_matching(CNN_PARAMS)

CONFIGS_A = {
    "MLP-S":      dict(kind="mlp", width=128,     permute=False),
    "MLP-M":      dict(kind="mlp", width=W_MATCH, permute=False),
    "MLP-L":      dict(kind="mlp", width=1024,    permute=False),
    "MLP-XL":     dict(kind="mlp", width=2048,    permute=False),
    "CNN":        dict(kind="cnn",                permute=False),
    "MLP-M-perm": dict(kind="mlp", width=W_MATCH, permute=True),
    "CNN-perm":   dict(kind="cnn",                permute=True),
}
ROLE = {"MLP-S": "capacity sweep", "MLP-M": "matched comparison", "MLP-L": "capacity sweep",
        "MLP-XL": "capacity sweep; reused in Q1(b)", "CNN": "matched comparison",
        "MLP-M-perm": "permutation control", "CNN-perm": "permutation control"}

def build_model(cfg):
    return MLP(cfg["width"]) if cfg["kind"] == "mlp" else CNN()

spec_rows = []
for name, cfg in CONFIGS_A.items():
    m = build_model(cfg)
    p = count_params(m)
    if cfg["kind"] == "mlp":
        assert p == mlp_param_count(cfg["width"]), "closed-form parameter count mismatch"
    spec_rows.append(dict(model=name, role=ROLE[name],
                          input="permuted pixels" if cfg["permute"] else "original",
                          specification=m.spec, parameters=p, params_vs_CNN=round(p / CNN_PARAMS, 3),
                          MACs_per_image=count_macs(m)))
SPEC_A = pd.DataFrame(spec_rows).set_index("model")
SPEC_A["MACs_per_param"] = (SPEC_A["MACs_per_image"] / SPEC_A["parameters"]).round(1)
SPEC_A.to_csv(os.path.join(OUT_DIR, "table_a1_model_specs.csv"))

STEPS_PER_EPOCH_A = math.ceil(N_TRAIN / CFG_A["batch_size"])
print(f"Shared budget: {CFG_A['epochs']} epochs x {STEPS_PER_EPOCH_A} steps = "
      f"{CFG_A['epochs'] * STEPS_PER_EPOCH_A} AdamW steps (batch {CFG_A['batch_size']}), cosine LR, "
      f"weight decay {CFG_A['weight_decay']}, no augmentation, LR grid {LR_GRID}, {N_SEEDS} seeds")
print(f"MLP-M width {W_MATCH}: {count_params(MLP(W_MATCH)):,} parameters vs CNN {CNN_PARAMS:,} "
      f"({100 * (count_params(MLP(W_MATCH)) / CNN_PARAMS - 1):+.2f}%)")
with pd.option_context("display.max_colwidth", 110):
    display(SPEC_A)
print(CNN())

In [ ]:
CACHE_TAG_A = hashlib.md5(json.dumps([SEED, N_TRAIN, N_VAL, CFG_A, TRAIN_EVAL_SUBSET],
                                     sort_keys=True).encode()).hexdigest()[:8]
TRAIN_EVAL_IDX = torch.from_numpy(np.random.default_rng(derive_seed("train-eval-subset")).choice(
    N_TRAIN, TRAIN_EVAL_SUBSET, replace=False)).to(DEVICE)

def run_a(name, lr, seed_idx):
    '''Train one Q1(a) model, or load it from the cache.'''
    path = os.path.join(OUT_DIR, "runs", f"q1a_{CACHE_TAG_A}_{name}_lr{lr:g}_s{seed_idx}.pt")
    if os.path.exists(path):
        out = torch.load(path, map_location="cpu")
    else:
        cfg = CONFIGS_A[name]
        seed = derive_seed("q1a-run", seed_idx)          # shared across models for a given seed index
        torch.manual_seed(seed)
        model = build_model(cfg)
        res = train_model(model, Xtr, ytr, Xva, yva, lr=lr, seed=seed,
                          input_fn=permute_pixels if cfg["permute"] else identity,
                          train_eval_idx=TRAIN_EVAL_IDX, tag=f"{name} lr={lr:g} seed#{seed_idx}", **CFG_A)
        out = dict(name=name, lr=lr, seed_idx=seed_idx, history=df_to_lists(res["history"]),
                   best_epoch=int(res["best_epoch"]), best_val_acc=float(res["best_val_acc"]),
                   best_state=res["best_state"], seconds=float(res["seconds"]))
        torch.save(out, path)
    out["history"] = pd.DataFrame(out["history"])
    return out

# ---- Learning-rate selection: same grid for every model, seed #0, validation accuracy only ----
T_START_A = time.time()
SWEEP = {}
for name in CONFIGS_A:
    for lr in LR_GRID:
        SWEEP[(name, lr)] = run_a(name, lr, seed_idx=0)

sweep_df = pd.DataFrame([dict(model=n, lr=lr, best_val_acc=r["best_val_acc"], best_epoch=r["best_epoch"])
                         for (n, lr), r in SWEEP.items()])
SELECTED_LR = {n: float(g.loc[g["best_val_acc"].idxmax(), "lr"]) for n, g in sweep_df.groupby("model", sort=False)}
LR_TABLE = sweep_df.pivot(index="model", columns="lr", values="best_val_acc").loc[list(CONFIGS_A)]
LR_TABLE.columns = [f"val acc @ lr={c:g}" for c in LR_TABLE.columns]
LR_TABLE["selected LR"] = pd.Series(SELECTED_LR)
LR_TABLE.to_csv(os.path.join(OUT_DIR, "table_a2_lr_selection.csv"))
print("Best-epoch VALIDATION accuracy per learning rate (seed #0). Used only to choose each model's LR:")
display(LR_TABLE.round(4))
edge = [n for n, lr in SELECTED_LR.items() if lr in (min(LR_GRID), max(LR_GRID))]
if edge:
    print(f"Note: selected LR is at the edge of the grid for {edge}; the grid is the same for every model, "
          "but consider extending it if time allows.")

In [ ]:
# ---- Repeat every model at its selected LR with further seeds ----
FINAL_RUNS_A = {}
for name in CONFIGS_A:
    lr = SELECTED_LR[name]
    FINAL_RUNS_A[name] = [SWEEP[(name, lr)]] + [run_a(name, lr, s) for s in range(1, N_SEEDS)]
for (name, lr), r in SWEEP.items():          # free checkpoints of unselected learning rates
    if lr != SELECTED_LR[name]:
        r["best_state"] = None
print(f"Q1(a) wall-clock this session: {(time.time() - T_START_A) / 60:.1f} min (cached runs load instantly)")
print("Mean training time per run (s):",
      {n: round(float(np.mean([r["seconds"] for r in runs])), 1) for n, runs in FINAL_RUNS_A.items()})

### Q1(a) training and validation curves
The solid lines are means over seeds and the shaded bands are ±1 sd. "Train" is the **eval-mode** accuracy and loss on a fixed 5k training
subset, so it is directly comparable with validation (BatchNorm running statistics, no dropout).

In [ ]:
def curve_stats(runs, col):
    M = np.stack([r["history"][col].values for r in runs])
    return M.mean(0), M.std(0)

def plot_curves(names, title, fname):
    fig, axes = plt.subplots(2, len(names), figsize=(3.7 * len(names), 6.2), sharex=True, squeeze=False)
    for j, name in enumerate(names):
        runs = FINAL_RUNS_A[name]
        ep = runs[0]["history"]["epoch"].values
        for i, (kind, ylab) in enumerate([("acc", "accuracy"), ("loss", "cross-entropy (log scale)")]):
            ax = axes[i, j]
            for col, lab, c in [(f"train_{kind}", "train (eval mode)", PALETTE[0]),
                                (f"val_{kind}", "validation", PALETTE[1])]:
                m, s = curve_stats(runs, col)
                ax.plot(ep, m, color=c, label=lab)
                ax.fill_between(ep, m - s, m + s, color=c, alpha=0.2, linewidth=0)
            best = np.mean([r["best_epoch"] for r in runs])
            ax.axvline(best, color=GREY, ls=":", lw=1)
            if kind == "loss":
                ax.set_yscale("log")
                ax.set_xlabel("epoch")
            else:
                ax.set_ylim(0, 1.02)
                ax.set_title(f"{name}: {SPEC_A.loc[name, 'parameters'] / 1e6:.2f}M params, "
                             f"LR {SELECTED_LR[name]:g}", fontsize=9)
            if j == 0:
                ax.set_ylabel(ylab)
    axes[0, 0].legend(loc="lower right", fontsize=8)
    axes[0, 0].text(0.02, 0.97, "dotted: mean best-val epoch", transform=axes[0, 0].transAxes,
                    fontsize=7, color=GREY, va="top")
    fig.suptitle(f"{title} (mean ± sd over {N_SEEDS} seeds)", y=1.0)
    fig.tight_layout()
    savefig(fig, fname)

plot_curves(["MLP-M", "CNN", "MLP-M-perm", "CNN-perm"],
            "Fig. A1: matched comparison and permutation controls", "fig_a1_curves_factorial.png")
plot_curves(["MLP-S", "MLP-M", "MLP-L", "MLP-XL"],
            "Fig. A2: MLP capacity sweep", "fig_a2_curves_capacity.png")

In [ ]:
# ---- Validation summary (used for the argument; test numbers come only in Section 5) ----
def at_best(run, col):
    h = run["history"]
    return float(h.loc[h["epoch"] == run["best_epoch"], col].iloc[0])

def sd(x):
    return float(np.std(x, ddof=1)) if len(x) > 1 else float("nan")

rows = []
for name, runs in FINAL_RUNS_A.items():
    va = np.array([r["best_val_acc"] for r in runs])
    rows.append(dict(model=name, parameters=int(SPEC_A.loc[name, "parameters"]), selected_lr=SELECTED_LR[name],
                     val_acc_mean=va.mean(), val_acc_sd=sd(va),
                     best_epoch_mean=np.mean([r["best_epoch"] for r in runs]),
                     train_acc_at_best=np.mean([at_best(r, "train_acc") for r in runs]),
                     train_acc_final=np.mean([r["history"]["train_acc"].iloc[-1] for r in runs]),
                     train_loss_final=np.mean([r["history"]["train_loss"].iloc[-1] for r in runs]),
                     val_loss_at_best=np.mean([at_best(r, "val_loss") for r in runs])))
SUMMARY_A = pd.DataFrame(rows).set_index("model")
SUMMARY_A["gap_at_best"] = SUMMARY_A["train_acc_at_best"] - SUMMARY_A["val_acc_mean"]
SUMMARY_A.to_csv(os.path.join(OUT_DIR, "table_a3_validation_summary.csv"))
print("Table A3: validation summary (best-validation checkpoint, mean over seeds)")
display(SUMMARY_A.round(4))

# ---- Fig. A3: capacity sweep and the 2x2 factorial ----
mlps = ["MLP-S", "MLP-M", "MLP-L", "MLP-XL"]
S = SUMMARY_A
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2), gridspec_kw=dict(width_ratios=[1.25, 1]))
P = S.loc[mlps, "parameters"].values.astype(float)
ax1.errorbar(P, S.loc[mlps, "val_acc_mean"], yerr=S.loc[mlps, "val_acc_sd"].fillna(0), marker="o", ms=7,
             capsize=3, color=PALETTE[0], label="MLP validation (best epoch)")
ax1.plot(P, S.loc[mlps, "train_acc_at_best"], marker="o", ms=7, ls="--", color=PALETTE[0], alpha=0.55,
         label="MLP train (eval mode) at that epoch")
for nm, mk, c in [("CNN", "*", PALETTE[1]), ("CNN-perm", "X", PALETTE[2])]:
    ax1.errorbar([S.loc[nm, "parameters"]], [S.loc[nm, "val_acc_mean"]],
                 yerr=[0 if np.isnan(S.loc[nm, "val_acc_sd"]) else S.loc[nm, "val_acc_sd"]],
                 marker=mk, ms=13, ls="none", capsize=3, color=c, label=f"{nm} validation")
for nm, x in zip(mlps, P):
    ax1.annotate(nm, (x, S.loc[nm, "val_acc_mean"]), textcoords="offset points", xytext=(0, -15),
                 ha="center", fontsize=8)
ax1.set_xscale("log")
ax1.set_ylim(0, 1.02)
ax1.set_xlabel("parameters (log scale)")
ax1.set_ylabel("accuracy")
ax1.set_title("Capacity: MLP width sweep vs. CNN at MLP-M's size", fontsize=10)
ax1.legend(fontsize=8, loc="center right")

bw = 0.36
for gi, (orig, perm) in enumerate([("MLP-M", "MLP-M-perm"), ("CNN", "CNN-perm")]):
    for k, (nm, lab, c) in enumerate([(orig, "original pixels", PALETTE[0]), (perm, "permuted pixels", PALETTE[1])]):
        vals = np.array([r["best_val_acc"] for r in FINAL_RUNS_A[nm]])
        x = gi + (k - 0.5) * (bw + 0.02)
        ax2.bar(x, vals.mean(), bw, color=c, edgecolor="white", linewidth=2, label=lab if gi == 0 else None)
        ax2.scatter(np.full(len(vals), x), vals, color="#0b0b0b", s=12, zorder=3)
        ax2.text(x, vals.mean() + 0.015, f"{vals.mean():.3f}", ha="center", fontsize=8)
ax2.set_xticks([0, 1])
ax2.set_xticklabels([f"MLP-M ({S.loc['MLP-M', 'parameters'] / 1e6:.2f}M)",
                     f"CNN ({S.loc['CNN', 'parameters'] / 1e6:.2f}M)"])
ax2.set_ylim(0, 1.15)
ax2.set_yticks(np.arange(0, 1.01, 0.2))
ax2.set_ylabel("validation accuracy (best epoch)")
ax2.set_title("Permutation control (dots = individual seeds)", fontsize=10)
ax2.legend(fontsize=8, loc="upper center", ncol=2)
fig.suptitle("Fig. A3: separating spatial inductive bias from capacity (validation)")
fig.tight_layout()
savefig(fig, "fig_a3_capacity_and_factorial.png")

In [ ]:
# ---- Quantitative tests of H1 / H2 / H3 on validation ----
def paired_diff(a_name, b_name):
    '''b - a, paired by seed index; returns mean, 95% t half-width, per-seed differences.'''
    a = np.array([r["best_val_acc"] for r in FINAL_RUNS_A[a_name]])
    b = np.array([r["best_val_acc"] for r in FINAL_RUNS_A[b_name]])
    d = b - a
    half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else float("nan")
    return float(d.mean()), float(half), d

E = {}
E["seed_sd"] = float(np.sqrt(np.nanmean([np.var([r["best_val_acc"] for r in runs], ddof=1)
                                          for runs in FINAL_RUNS_A.values()]))) if N_SEEDS > 1 else float("nan")
# H3 / H2 (underfitting): does the matched MLP fit its own training data?
E["mlp_train_final"] = float(S.loc["MLP-M", "train_acc_final"])
E["mlp_train_loss_final"] = float(S.loc["MLP-M", "train_loss_final"])
E["mlp_val"] = float(S.loc["MLP-M", "val_acc_mean"])
E["mlp_gap"] = float(S.loc["MLP-M", "gap_at_best"])
E["cnn_train_final"] = float(S.loc["CNN", "train_acc_final"])
E["cnn_val"] = float(S.loc["CNN", "val_acc_mean"])
# H2 (capacity): validation gain from widening the MLP
logp = np.log2(S.loc[mlps, "parameters"].values.astype(float))
E["capacity_slope_per_doubling"] = float(np.polyfit(logp, S.loc[mlps, "val_acc_mean"].values, 1)[0])
E["capacity_gain_S_to_XL"] = float(S.loc["MLP-XL", "val_acc_mean"] - S.loc["MLP-S", "val_acc_mean"])
E["param_ratio_S_to_XL"] = float(S.loc["MLP-XL", "parameters"] / S.loc["MLP-S", "parameters"])
E["xl_train_final"] = float(S.loc["MLP-XL", "train_acc_final"])
E["cnn_minus_mlp"], E["cnn_minus_mlp_ci"], _ = paired_diff("MLP-M", "CNN")
E["doublings_needed"] = (E["cnn_minus_mlp"] / E["capacity_slope_per_doubling"]
                         if E["capacity_slope_per_doubling"] > 0 else float("inf"))
# H1 (spatial structure): permutation effects and their interaction
E["perm_effect_mlp"], E["perm_effect_mlp_ci"], _ = paired_diff("MLP-M", "MLP-M-perm")
E["perm_effect_cnn"], E["perm_effect_cnn_ci"], _ = paired_diff("CNN", "CNN-perm")
E["cnnperm_minus_mlpperm"], E["cnnperm_minus_mlpperm_ci"], _ = paired_diff("MLP-M-perm", "CNN-perm")
E["interaction"] = E["perm_effect_cnn"] - E["perm_effect_mlp"]

def ci(x):
    return "" if np.isnan(x) else f" ± {100 * x:.1f}"

EVIDENCE_A = pd.DataFrame([
    ("MLP-M train acc (eval, final epoch) / val acc", f"{pct(E['mlp_train_final'])} / {pct(E['mlp_val'])}",
     "H1: high train, big gap | H2/H3: low train"),
    ("MLP-M train loss (eval, final epoch)", f"{E['mlp_train_loss_final']:.3f}", "H3: stays high"),
    (f"Val gain MLP-S -> MLP-XL ({E['param_ratio_S_to_XL']:.0f}x params)", pp(E["capacity_gain_S_to_XL"]),
     "H2: large | H1: small"),
    ("Val gain per doubling of MLP params", pp(E["capacity_slope_per_doubling"]), "H2: large"),
    ("CNN - MLP-M at equal params (paired, 95% CI)", pp(E["cnn_minus_mlp"]) + ci(E["cnn_minus_mlp_ci"]),
     "H1: large | H2: ~0"),
    ("Permutation effect on MLP-M", pp(E["perm_effect_mlp"]) + ci(E["perm_effect_mlp_ci"]), "all: ~0"),
    ("Permutation effect on CNN", pp(E["perm_effect_cnn"]) + ci(E["perm_effect_cnn_ci"]),
     "H1: large negative | H2/H3: ~0"),
    ("CNN-perm - MLP-M-perm", pp(E["cnnperm_minus_mlpperm"]) + ci(E["cnnperm_minus_mlpperm_ci"]),
     "H1: ~0 or negative"),
    ("Interaction (CNN perm effect - MLP perm effect)", pp(E["interaction"]), "H1: large negative | H2: ~0"),
    ("Seed-to-seed sd of val acc (pooled)", "n/a" if np.isnan(E["seed_sd"]) else f"{100 * E['seed_sd']:.2f} pp",
     "noise scale"),
], columns=["measurement (validation)", "value", "prediction"])
EVIDENCE_A.to_csv(os.path.join(OUT_DIR, "table_a4_evidence.csv"), index=False)
with pd.option_context("display.max_colwidth", 80):
    display(EVIDENCE_A)

## 4. Q1(b): Does the MLP learn useful structure before it memorises?

### Setting
**40% of the 45k training labels are replaced** by a class drawn uniformly from the nine *wrong* classes. The draw is seeded
from the roll number, and the corrupted labels stay fixed for the whole run. A corrupted label is, by construction, unrelated to
its image, so fitting it cannot generalise and can only be memorised. Validation labels stay clean, so validation
accuracy measures generalisation to the true task.

### Model and recipe (reused from Q1(a))
* **Architecture: MLP-XL** from the Q1(a) capacity sweep (width 2048, ≈14.7M parameters, about 330 per training example). It is the largest MLP in this notebook, so its capacity to memorise 18k arbitrary labels is not in doubt, and the run checks this by measuring the final fit of the corrupted labels.
* **Reused unchanged:** `MLP` class, `train_model` loop, AdamW, batch size 128, the data split and normalisation.
* **Changed on purpose:** **no weight decay, a constant LR and 150 epochs.** Weight decay and early stopping are anti-memorisation devices, and we want a setting where memorisation *can* happen. A decaying LR would add its own phase changes to the trajectory, making memorisation onset impossible to separate from a schedule effect. LR 1e-3 is the middle of the Q1(a) grid and is not tuned, because the target here is the training dynamics, not the best score.
* **Control:** an identical run on **clean** labels (same initialisation and batch order).

### Measurements along the trajectory
These are taken every quarter epoch for the first 10 epochs, then every epoch, on the whole training set in eval mode:
* **clean-train accuracy:** accuracy on the 60% of examples whose labels are correct;
* **memorisation rate:** the fraction of corrupted examples predicted as their **assigned wrong label**;
* **shadow rate (null):** the fraction predicted as a **second wrong label drawn the same way but never shown to the model**. Without example-specific memorisation the assigned and shadow labels are exchangeable, so the two rates match. **Memorisation excess = memorisation rate − shadow rate**, with a paired z-score;
* **true-label accuracy on corrupted examples:** a network that has learned class structure predicts the *real* class of a mislabelled image even though it is being trained to say otherwise (chance is 10%);
* validation accuracy, and cross-entropy on the clean and the corrupted subsets;
* **per example:** the epoch from which it stays fitted to its assigned label until the end of training ("stably learned").

### Decision rule (fixed in advance)
* **Memorisation onset** `t_mem` is the first evaluation point after which the memorisation excess stays significantly positive (z > 3) for the rest of training.
* **Peak** `t_peak` is the evaluation point of maximum validation accuracy.
* We conclude that **useful structure is learned before memorisation begins** if both of these hold at `t_mem`:
  1. validation accuracy has already covered at least half of its eventual rise above chance (10% → peak);
  2. true-label accuracy on the corrupted examples is above 20% (twice chance).

  Supporting evidence would be that validation accuracy and true-label accuracy on corrupted examples **fall** as the memorisation excess rises, and that clean examples become stably learned much earlier than corrupted ones, while the same images *with their true labels* (control run) are learned early.

In [ ]:
rng_noise = np.random.default_rng(derive_seed("label-noise", NOISE_RATE))
y_true_np = y_all[train_idx]
N_NOISY = int(round(NOISE_RATE * N_TRAIN))
NOISY_MASK = np.zeros(N_TRAIN, dtype=bool)
NOISY_MASK[rng_noise.choice(N_TRAIN, N_NOISY, replace=False)] = True
y_noisy_np = y_true_np.copy()
y_noisy_np[NOISY_MASK] = (y_true_np[NOISY_MASK] + rng_noise.integers(1, 10, N_NOISY)) % 10
y_shadow_np = y_true_np.copy()                       # never shown to any model
y_shadow_np[NOISY_MASK] = (y_true_np[NOISY_MASK] + rng_noise.integers(1, 10, N_NOISY)) % 10
assert (y_noisy_np[NOISY_MASK] != y_true_np[NOISY_MASK]).all()
assert (y_shadow_np[NOISY_MASK] != y_true_np[NOISY_MASK]).all()

print(f"{N_NOISY:,} of {N_TRAIN:,} training labels corrupted ({NOISE_RATE:.0%}); validation labels untouched.")
print(f"assigned == shadow label on {np.mean(y_noisy_np[NOISY_MASK] == y_shadow_np[NOISY_MASK]):.3f} "
      f"of corrupted examples (expected 1/9 = 0.111)")
display(pd.DataFrame({"corrupted": np.bincount(y_true_np[NOISY_MASK], minlength=10),
                      "total": np.bincount(y_true_np, minlength=10)}, index=CLASSES).T)

In [ ]:
MEM_MODEL = "MLP-XL"      # architecture reused from Q1(a)
CACHE_TAG_B = hashlib.md5(json.dumps([SEED, N_TRAIN, CFG_B, NOISE_RATE, DENSE_EVAL_EPOCHS_B,
                                      CONFIGS_A[MEM_MODEL]], sort_keys=True).encode()).hexdigest()[:8]

class MemorisationTracker:
    '''Evaluates the whole training set (eval mode) and the validation set at chosen steps.
    `subset` = corrupted examples (in the clean control: the same images, with their true labels).'''
    def __init__(self, y_assigned, subset_mask):
        self.y_true = torch.from_numpy(y_true_np).to(DEVICE)
        self.y_assigned = torch.from_numpy(y_assigned).to(DEVICE)
        self.y_shadow = torch.from_numpy(y_shadow_np).to(DEVICE)
        self.sub = torch.from_numpy(subset_mask).to(DEVICE)
        self.rest = ~self.sub
        self.n_sub = int(subset_mask.sum())
        self.records, self.fit = [], []
        self.best = dict(val_acc=-1.0, epoch=0.0, state=None)

    @torch.no_grad()
    def __call__(self, model, epoch):
        logits = predict_logits(model, Xtr)
        pred = logits.argmax(1)
        ce = F.cross_entropy(logits, self.y_assigned, reduction="none")
        v_loss, v_acc = loss_acc(predict_logits(model, Xva), yva)
        s, r = self.sub, self.rest
        hit = pred == self.y_assigned
        hit_shadow = pred == self.y_shadow
        d = hit[s].float() - hit_shadow[s].float()
        se = float(d.std()) / math.sqrt(max(self.n_sub, 1))
        rec = dict(epoch=float(epoch), val_acc=v_acc, val_loss=v_loss,
                   train_fit=hit.float().mean().item(),
                   clean_acc=(pred[r] == self.y_true[r]).float().mean().item(),
                   mem_rate=hit[s].float().mean().item(),
                   shadow_rate=hit_shadow[s].float().mean().item(),
                   true_acc_on_corrupted=(pred[s] == self.y_true[s]).float().mean().item(),
                   loss_clean=ce[r].mean().item(), loss_corrupted=ce[s].mean().item())
        rec["mem_excess"] = rec["mem_rate"] - rec["shadow_rate"]
        rec["mem_z"] = rec["mem_excess"] / se if se > 0 else 0.0
        self.records.append(rec)
        self.fit.append(hit.cpu())
        if v_acc > self.best["val_acc"]:
            self.best = dict(val_acc=v_acc, epoch=float(epoch), state=cpu_state(model))

def run_b(noise_rate):
    path = os.path.join(OUT_DIR, "runs", f"q1b_{CACHE_TAG_B}_noise{noise_rate:g}.pt")
    if os.path.exists(path):
        return torch.load(path, map_location="cpu")
    y_assigned = y_noisy_np if noise_rate > 0 else y_true_np
    seed = derive_seed("q1b-run")          # same initialisation and batch order for noisy run and control
    torch.manual_seed(seed)
    model = build_model(CONFIGS_A[MEM_MODEL]).to(DEVICE)
    tracker = MemorisationTracker(y_assigned, NOISY_MASK)
    spe = math.ceil(N_TRAIN / CFG_B["batch_size"])
    steps = sorted({max(1, round(k * spe / 4)) for k in range(1, 4 * DENSE_EVAL_EPOCHS_B + 1)}
                   | {e * spe for e in range(1, CFG_B["epochs"] + 1)})
    tracker(model, 0.0)                    # initialisation
    res = train_model(model, Xtr, torch.from_numpy(y_assigned).to(DEVICE), Xva, yva, seed=seed,
                      step_callback=tracker, callback_steps=steps,
                      tag=f"{MEM_MODEL} noise={noise_rate:g}", print_every=10, **CFG_B)
    out = dict(noise_rate=float(noise_rate),
               traj={k: [float(rec[k]) for rec in tracker.records] for k in tracker.records[0]},
               fit=torch.stack(tracker.fit), best_state=tracker.best["state"],
               best_epoch=float(tracker.best["epoch"]), best_val_acc=float(tracker.best["val_acc"]),
               final_state=res["final_state"], seconds=float(res["seconds"]))
    torch.save(out, path)
    return out

t0 = time.time()
RUN_NOISY = run_b(NOISE_RATE)
RUN_CLEAN = run_b(0.0)
TRAJ_N, TRAJ_C = pd.DataFrame(RUN_NOISY["traj"]), pd.DataFrame(RUN_CLEAN["traj"])
print(f"Q1(b) wall-clock this session: {(time.time() - t0) / 60:.1f} min; "
      f"{len(TRAJ_N)} evaluation points per run")

In [ ]:
def stays_true_from(flags):
    '''Index of the first element from which every later flag is True (None if the last one is False).'''
    flags = np.asarray(flags, bool)
    tail = np.flip(np.cumprod(np.flip(flags))).astype(bool)
    return int(np.argmax(tail)) if tail.any() else None

def stable_epochs(run):
    '''Per example: first evaluation epoch from which it stays fitted to its assigned label (inf if never).'''
    fit = run["fit"].numpy()
    ep = np.asarray(run["traj"]["epoch"])
    tail = np.flip(np.cumprod(np.flip(fit, 0), 0), 0).astype(bool)
    learned = tail.any(0)
    return np.where(learned, ep[np.where(learned, tail.argmax(0), 0)], np.inf)

T = TRAJ_N
i_mem = stays_true_from(T["mem_z"].values > 3)
i_peak = int(T["val_acc"].idxmax())
i_end = len(T) - 1
B = dict(t_peak=float(T.epoch[i_peak]), val_peak=float(T.val_acc[i_peak]),
         clean_acc_peak=float(T.clean_acc[i_peak]), true_corr_peak=float(T.true_acc_on_corrupted[i_peak]),
         excess_peak=float(T.mem_excess[i_peak]), mem_rate_peak=float(T.mem_rate[i_peak]),
         t_end=float(T.epoch[i_end]), val_end=float(T.val_acc[i_end]), clean_acc_end=float(T.clean_acc[i_end]),
         mem_rate_end=float(T.mem_rate[i_end]), shadow_end=float(T.shadow_rate[i_end]),
         excess_end=float(T.mem_excess[i_end]), true_corr_end=float(T.true_acc_on_corrupted[i_end]),
         true_corr_max=float(T.true_acc_on_corrupted.max()),
         t_true_corr_max=float(T.epoch[T.true_acc_on_corrupted.idxmax()]),
         clean_val_peak=float(TRAJ_C.val_acc.max()), clean_val_end=float(TRAJ_C.val_acc.iloc[-1]),
         t_mem=None)
if i_mem is not None:
    B.update(t_mem=float(T.epoch[i_mem]), val_at_mem=float(T.val_acc[i_mem]),
             true_corr_at_mem=float(T.true_acc_on_corrupted[i_mem]), clean_acc_at_mem=float(T.clean_acc[i_mem]),
             frac_of_rise_at_mem=float((T.val_acc[i_mem] - 0.1) / max(B["val_peak"] - 0.1, 1e-9)))
    B["structure_first"] = bool(B["frac_of_rise_at_mem"] >= 0.5 and B["true_corr_at_mem"] > 0.2)
else:
    B["structure_first"] = None
post = T.iloc[i_peak:]
B["rho_excess_val_post_peak"] = (float(stats.spearmanr(post.mem_excess, post.val_acc)[0])
                                 if len(post) > 2 else float("nan"))

# ---- Table B1: trajectory at milestones ----
targets = [0, 0.25, 0.5, 1, 2, 3, 5, 10, 20, 50, 100, T.epoch.max()]
idx = sorted({int((T.epoch - t).abs().idxmin()) for t in targets} | {i_peak} | ({i_mem} if i_mem is not None else set()))
cols = ["epoch", "val_acc", "clean_acc", "true_acc_on_corrupted", "mem_rate", "shadow_rate", "mem_excess",
        "mem_z", "loss_clean", "loss_corrupted"]
TABLE_B1 = T.loc[idx, cols].copy()
TABLE_B1.insert(2, "val_acc_clean_control", TRAJ_C.loc[idx, "val_acc"].values)
TABLE_B1["event"] = [" + ".join(([ "peak val acc"] if i == i_peak else []) +
                                 (["memorisation onset"] if i == i_mem else [])) for i in idx]
TABLE_B1.to_csv(os.path.join(OUT_DIR, "table_b1_memorisation_trajectory.csv"), index=False)
print(f"Table B1: MLP-XL, {NOISE_RATE:.0%} corrupted labels (training set measured in eval mode)")
display(TABLE_B1.round(4).set_index("epoch"))

# ---- Table B2: when do examples become stably fitted? ----
st_n, st_c = stable_epochs(RUN_NOISY), stable_epochs(RUN_CLEAN)
groups = {"clean examples (noisy run)": st_n[~NOISY_MASK],
          "corrupted examples (noisy run)": st_n[NOISY_MASK],
          "same images, true labels (clean control)": st_c[NOISY_MASK],
          "other images (clean control)": st_c[~NOISY_MASK]}
TABLE_B2 = pd.DataFrame([dict(group=g, n=len(v), median_stable_epoch=float(np.median(v)),
                              stably_fitted_by_peak=float(np.mean(v <= B["t_peak"])),
                              stably_fitted_by_end=float(np.mean(np.isfinite(v))))
                         for g, v in groups.items()]).set_index("group")
TABLE_B2.to_csv(os.path.join(OUT_DIR, "table_b2_stable_learning_time.csv"))
B["median_stable"] = TABLE_B2["median_stable_epoch"].to_dict()
print("Table B2: per-example learning time (inf = not stably fitted by the end)")
display(TABLE_B2.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
ax = axes[0]
x = T.epoch.values
ax.plot(x, T.val_acc, color=PALETTE[0], label="validation acc")
ax.plot(x, TRAJ_C.val_acc, color=PALETTE[0], ls="--", lw=1.3, alpha=0.7, label="validation acc, clean-label control")
ax.plot(x, T.clean_acc, color=PALETTE[1], label="train acc, clean examples")
ax.plot(x, T.true_acc_on_corrupted, color=PALETTE[2], label="corrupted examples: TRUE class predicted")
ax.plot(x, T.mem_rate, color=PALETTE[6], label="corrupted examples: ASSIGNED label (memorisation)")
ax.plot(x, T.shadow_rate, color=GREY, ls=":", lw=1.5, label="corrupted examples: shadow label (null)")
ax.set_ylabel("fraction")
ax.set_title(f"Fig. B1a: MLP-XL, {NOISE_RATE:.0%} corrupted labels", fontsize=10)
ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)

ax = axes[1]
ax.plot(x, T.loss_clean, color=PALETTE[1], label="clean examples")
ax.plot(x, T.loss_corrupted, color=PALETTE[6], label="corrupted examples (w.r.t. assigned label)")
ax.plot(x, T.val_loss, color=PALETTE[0], label="validation")
ax.set_yscale("log")
ax.set_ylabel("cross-entropy (log scale)")
ax.set_title("Fig. B1b: losses", fontsize=10)
ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)

ax = axes[2]
for (g, v), c, ls in zip(groups.items(), [PALETTE[1], PALETTE[6], PALETTE[2], GREY], ["-", "-", "--", ":"]):
    finite = np.sort(v[np.isfinite(v)])
    ax.step(np.r_[0, finite], np.r_[0, np.arange(1, len(finite) + 1) / len(v)], where="post",
            color=c, ls=ls, label=g)
ax.set_ylim(0, 1.02)
ax.set_ylabel("fraction stably fitted")
ax.set_title("Fig. B1c: when examples become stably fitted", fontsize=10)
ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)

for ax in axes:
    ax.set_xscale("symlog", linthresh=1)
    ax.set_xlim(0, T.epoch.max())
    ax.set_xlabel("epoch (linear below 1, log above)")
    ax.axvline(B["t_peak"], color=GREY, lw=1)
    if B["t_mem"] is not None:
        ax.axvline(B["t_mem"], color=PALETTE[7], lw=1, ls="--")
axes[0].text(B["t_peak"], 1.0, " peak val", color=GREY, fontsize=7, va="top")
if B["t_mem"] is not None:
    axes[0].text(B["t_mem"], 0.93, " memorisation onset", color=PALETTE[7], fontsize=7, va="top")
fig.tight_layout()
savefig(fig, "fig_b1_memorisation.png")

## 5. Final test-set evaluation (run once, after every choice is fixed)

At this point the models, learning rates and early-stopping checkpoints have all been chosen on validation. The official
CIFAR-10 test set is now loaded for the first time and used only to report results; nothing below feeds back into any choice.
Test images are normalised with the training-split statistics. The permuted conditions see the test images through the same
fixed permutation.

In [ ]:
test_ds = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=True)
Xte = to_tensor(np.asarray(test_ds.data)).to(DEVICE)
yte = torch.from_numpy(np.asarray(test_ds.targets, dtype=np.int64)).to(DEVICE)

def test_accuracy(cfg, state, input_fn):
    model = build_model(cfg).to(DEVICE)
    model.load_state_dict(state)
    return loss_acc(predict_logits(model, Xte, input_fn), yte)[1]

TEST_A = {}
rows = []
for name, runs in FINAL_RUNS_A.items():
    cfg = CONFIGS_A[name]
    accs = np.array([test_accuracy(cfg, r["best_state"], permute_pixels if cfg["permute"] else identity)
                     for r in runs])
    TEST_A[name] = accs
    rows.append(dict(model=name, parameters=int(SPEC_A.loc[name, "parameters"]), selected_lr=SELECTED_LR[name],
                     val_acc_mean=SUMMARY_A.loc[name, "val_acc_mean"], test_acc_mean=accs.mean(),
                     test_acc_sd=sd(accs), test_acc_per_seed=", ".join(f"{a:.4f}" for a in accs)))
FINAL_TEST_A = pd.DataFrame(rows).set_index("model")
FINAL_TEST_A.to_csv(os.path.join(OUT_DIR, "table_final_test_q1a.csv"))
print("Table T1: Q1(a) FINAL TEST ACCURACY (best-validation checkpoint of each seed)")
display(FINAL_TEST_A.round(4))

def test_diff(a, b):
    return float((TEST_A[b] - TEST_A[a]).mean())
E_TEST = dict(cnn_minus_mlp=test_diff("MLP-M", "CNN"), perm_effect_mlp=test_diff("MLP-M", "MLP-M-perm"),
              perm_effect_cnn=test_diff("CNN", "CNN-perm"), cnnperm_minus_mlpperm=test_diff("MLP-M-perm", "CNN-perm"),
              capacity_gain_S_to_XL=test_diff("MLP-S", "MLP-XL"))
E_TEST["interaction"] = E_TEST["perm_effect_cnn"] - E_TEST["perm_effect_mlp"]
display(pd.Series({k: pp(v) for k, v in E_TEST.items()}, name="test-set differences (paired by seed)").to_frame())

In [ ]:
rows = []
for label, run in [(f"{NOISE_RATE:.0%} corrupted labels", RUN_NOISY), ("clean-label control", RUN_CLEAN)]:
    traj = pd.DataFrame(run["traj"])
    for ckpt, state, ep in [("early-stopped (best validation)", run["best_state"], run["best_epoch"]),
                            ("final epoch", run["final_state"], traj.epoch.iloc[-1])]:
        k = int((traj.epoch - ep).abs().idxmin())
        rows.append(dict(run=label, checkpoint=ckpt, epoch=ep, val_acc=traj.val_acc[k],
                         train_fit_assigned=traj.train_fit[k],
                         corrupted_fit=traj.mem_rate[k] if run is RUN_NOISY else float("nan"),
                         test_acc=test_accuracy(CONFIGS_A[MEM_MODEL], state, identity)))
FINAL_TEST_B = pd.DataFrame(rows)
FINAL_TEST_B.to_csv(os.path.join(OUT_DIR, "table_final_test_q1b.csv"), index=False)
print("Table T2: Q1(b) TEST ACCURACY: model before memorisation vs. after")
display(FINAL_TEST_B.round(4))

## 6. Interpretation of the evidence

The next cell writes the interpretation **from the measured numbers**, applying the predictions and the decision rule
stated before the experiments were run. The text is also saved to `q1_outputs/interpretation.txt`, and you can paste it into the report.

In [ ]:
A_TEXT = []
fits = E["mlp_train_final"] >= 0.90
A_TEXT.append(
    f"Optimisation / underfitting (H3, H2): the parameter-matched MLP-M reaches {pct(E['mlp_train_final'])} training "
    f"accuracy (eval mode, final epoch; training loss {E['mlp_train_loss_final']:.3f}) but {pct(E['mlp_val'])} validation "
    f"accuracy at its best epoch. " +
    ("It fits its own training data, so its poor CIFAR-10 accuracy is a generalisation failure, not a failure of "
     "optimisation or of capacity to fit." if fits else
     "It does not fully fit the training set within the budget, so an optimisation or capacity contribution cannot be "
     "excluded from this measurement alone."))
cap_small = E["cnn_minus_mlp"] > 0 and E["capacity_gain_S_to_XL"] < 0.5 * E["cnn_minus_mlp"]
dn = E["doublings_needed"]
A_TEXT.append(
    f"Capacity (H2): widening the MLP {E['param_ratio_S_to_XL']:.0f}x (MLP-S to MLP-XL) changes validation accuracy by "
    f"{pp(E['capacity_gain_S_to_XL'])} ({pp(E['capacity_slope_per_doubling'])} per doubling of parameters), while MLP-XL "
    f"fits {pct(E['xl_train_final'])} of its training data. The CNN with MLP-M's parameter count differs from "
    f"MLP-M by {pp(E['cnn_minus_mlp'])}. " +
    (f"At the observed trend the MLP would need about {dn:.0f} more doublings of parameters (x{2 ** min(dn, 60):.1e}) "
     f"to close that gap. " if np.isfinite(dn) and dn > 0 else
     ("Extra width does not move the MLP towards the CNN at all. " if not np.isfinite(dn) else "")) +
    ("Capacity does not explain the gap." if cap_small else
     ("There is no CNN advantage to explain." if E["cnn_minus_mlp"] <= 0 else
      "Capacity accounts for a substantial part of the gap.")))
band = max(2 * E["seed_sd"], 0.01) if not np.isnan(E["seed_sd"]) else 0.01
mlp_inv = abs(E["perm_effect_mlp"]) <= band
cnn_collapse = E["cnnperm_minus_mlpperm"] <= 0.5 * E["cnn_minus_mlp"]
A_TEXT.append(
    f"Spatial inductive bias (H1): permuting pixels changes MLP-M by {pp(E['perm_effect_mlp'])} "
    + ("(within run-to-run noise, as expected for a permutation-invariant model)" if mlp_inv else
       "(more than run-to-run noise, unexpected for a permutation-invariant model; inspect the runs)")
    + f" and the CNN by {pp(E['perm_effect_cnn'])}. The CNN's advantage over the MLP goes from {pp(E['cnn_minus_mlp'])} on "
    f"real images to {pp(E['cnnperm_minus_mlpperm'])} on permuted images (interaction {pp(E['interaction'])}), with "
    f"parameters, FLOPs, data, optimiser and tuning effort held fixed.")
if fits and cap_small and mlp_inv and cnn_collapse:
    verdict_a = ("Conclusion: the evidence supports H1 and is inconsistent with H2 and H3. The MLP is not short of "
                 "capacity and is not hard to optimise: it fits the training set but generalises poorly. A "
                 "model of the same size and training budget that builds in locality and weight sharing does much better, "
                 "and loses that advantage when the spatial structure is removed. The MLP's weakness on CIFAR-10 is "
                 "therefore its lack of a spatial inductive bias.")
else:
    verdict_a = ("Conclusion: the pattern does not match H1's predictions cleanly. Checks that failed: " +
                 ", ".join(n for n, ok in [("MLP fits training data", fits), ("capacity gain small", cap_small),
                                           ("MLP permutation-invariant", mlp_inv),
                                           ("CNN advantage removed by permutation", cnn_collapse)] if not ok) +
                 ". Report the competing explanation(s) these results leave open.")
A_TEXT.append(verdict_a)
A_TEXT.append(
    f"Test set (evaluated once after all choices): MLP-M {pct(FINAL_TEST_A.loc['MLP-M', 'test_acc_mean'])}, "
    f"CNN {pct(FINAL_TEST_A.loc['CNN', 'test_acc_mean'])}, MLP-M-perm {pct(FINAL_TEST_A.loc['MLP-M-perm', 'test_acc_mean'])}, "
    f"CNN-perm {pct(FINAL_TEST_A.loc['CNN-perm', 'test_acc_mean'])}; MLP-S/L/XL "
    f"{pct(FINAL_TEST_A.loc['MLP-S', 'test_acc_mean'])}/{pct(FINAL_TEST_A.loc['MLP-L', 'test_acc_mean'])}/"
    f"{pct(FINAL_TEST_A.loc['MLP-XL', 'test_acc_mean'])}. Test-set interaction {pp(E_TEST['interaction'])} "
    f"(validation {pp(E['interaction'])}), so the conclusion "
    + ("carries over to unseen data." if np.sign(E_TEST["interaction"]) == np.sign(E["interaction"]) else
       "does NOT replicate in sign on the test set; treat it with caution."))

B_TEXT = []
B_TEXT.append(
    f"Capacity to memorise: by epoch {B['t_end']:g}, MLP-XL predicts the assigned wrong label for {pct(B['mem_rate_end'])} "
    f"of the {N_NOISY:,} corrupted examples (shadow-label null {pct(B['shadow_end'])}), and fits {pct(B['clean_acc_end'])} "
    f"of the clean ones, so the setting does let the MLP fit examples that cannot generalise.")
B_TEXT.append(
    f"Early phase: at the validation peak (epoch {B['t_peak']:g}) validation accuracy is {pct(B['val_peak'])}, clean-train "
    f"accuracy is {pct(B['clean_acc_peak'])}, and the network predicts the TRUE class for {pct(B['true_corr_peak'])} of the "
    f"mislabelled training images (chance 10%), even though it is trained on their wrong labels. The memorisation excess "
    f"is only {pp(B['excess_peak'])} at this point.")
if B["t_mem"] is not None:
    B_TEXT.append(
        f"Memorisation onset (excess significant from then on, z > 3): epoch {B['t_mem']:g}. By then validation accuracy "
        f"had covered {B['frac_of_rise_at_mem']:.0%} of its rise from chance to its peak, and true-label accuracy on the "
        f"corrupted examples was {pct(B['true_corr_at_mem'])}.")
else:
    B_TEXT.append("The memorisation excess never becomes significantly and persistently positive in this run.")
B_TEXT.append(
    f"Late phase: from the peak to epoch {B['t_end']:g}, the memorisation excess rises to {pp(B['excess_end'])}, while "
    f"validation accuracy changes by {pp(B['val_end'] - B['val_peak'])} to {pct(B['val_end'])} and true-label accuracy on "
    f"corrupted examples goes from a maximum of {pct(B['true_corr_max'])} (epoch {B['t_true_corr_max']:g}) to "
    f"{pct(B['true_corr_end'])}. After the peak, the Spearman correlation between the memorisation excess and validation "
    f"accuracy is {B['rho_excess_val_post_peak']:.2f}. The clean-label control with the identical recipe peaks at "
    f"{pct(B['clean_val_peak'])} and ends at {pct(B['clean_val_end'])}, so the extra decline in the noisy run comes from fitting "
    f"the corrupted labels.")
ms = B["median_stable"]
fmt_ep = lambda v: "never (median not reached)" if not np.isfinite(v) else f"epoch {v:g}"
B_TEXT.append(
    f"Per example: the median clean example is stably fitted from {fmt_ep(ms['clean examples (noisy run)'])}, the median "
    f"corrupted example from {fmt_ep(ms['corrupted examples (noisy run)'])}. The same images with their true labels "
    f"(control run) are fitted from {fmt_ep(ms['same images, true labels (clean control)'])}, so the delay is caused by "
    f"the label being inconsistent with the image, not by the images being harder.")
nt = FINAL_TEST_B[FINAL_TEST_B.run.str.contains("corrupted")].set_index("checkpoint")
B_TEXT.append(
    f"Test set: the early-stopped checkpoint (epoch {nt.loc['early-stopped (best validation)', 'epoch']:g}, chosen on "
    f"validation) scores {pct(nt.loc['early-stopped (best validation)', 'test_acc'])}, against "
    f"{pct(nt.loc['final epoch', 'test_acc'])} for the final memorising model.")
if B["structure_first"] is True:
    verdict_b = ("Decision: YES, useful structure is learned before memorisation begins. Generalising, class-level "
                 "structure (rising validation accuracy, and the true class predicted for mislabelled images) appears "
                 "while the fit to the corrupted labels is still at the shadow-label null. Memorisation of the "
                 "corrupted labels starts later and then erodes that structure, as validation accuracy and true-label "
                 "accuracy on the corrupted examples fall together while the memorisation excess grows.")
elif B["structure_first"] is False:
    verdict_b = ("Decision: NO, by the pre-registered rule. Memorisation of the corrupted labels becomes significant "
                 "before the network has learned most of its generalising structure, so the two phases overlap "
                 "rather than occurring in sequence in this setting.")
else:
    verdict_b = "Decision: undetermined, because no persistent memorisation was detected (train longer or use more capacity)."
B_TEXT.append(verdict_b)

report = "Q1(a)\n" + "\n\n".join(A_TEXT) + "\n\nQ1(b)\n" + "\n\n".join(B_TEXT)
with open(os.path.join(OUT_DIR, "interpretation.txt"), "w") as f:
    f.write(report)
print(report)

### Limitations to state in the report
* **Absolute accuracies are below standard CIFAR-10 numbers by design.** There is no augmentation and the budget is a short 30-epoch run, because the aim is a controlled comparison, not a leaderboard score.
* **One fixed permutation** is used. Any fixed permutation is equivalent for the MLP, and for the CNN a different random permutation would destroy locality to the same degree.
* **FLOPs are not matched** between MLP and CNN (see `MACs_per_image`). The permutation control is what rules compute out as the explanation.
* **Seed variation:** `N_SEEDS` runs per model. Differences are reported next to the seed-to-seed sd and paired 95% t-intervals, which are wide with only 3 seeds.
* **Q1(b) uses one noise rate (40%) and one architecture.** The memorisation onset depends on the learning rate and the model size, so the claim concerns ordering (structure first, memorisation later) and not the exact epochs.
* The LR for each Q1(a) model is picked on seed #0, which is then reused as one of the reported seeds. This small optimistic bias applies equally to every model.

## 7. Reuse log

| Component | Introduced in | Reused in | Changes on reuse |
|---|---|---|---|
| Stratified 45k/5k split seeded by the roll number, normalisation statistics (§1) | Q1 setup | Q1(a), Q1(b); indices saved in `split_indices.npz` for later questions | none |
| `MLP`, `CNN`, `count_params`, `count_macs` (§2) | Q1 shared components | Q1(a), Q1(b) | none |
| `train_model`, `predict_logits` (§2) | Q1 shared components | Q1(a), Q1(b) | Q1(b) passes a step callback for dense evaluation |
| **MLP-XL** architecture (width 2048, 14.7M params) | Q1(a) capacity sweep | Q1(b) memorisation model | recipe changed: no weight decay, constant LR 1e-3, 150 epochs, 40% corrupted labels |
| Q1(a) results (validation curves, selected LRs, checkpoints) | Q1(a) | §5 test evaluation, §6 interpretation | none |

In [ ]:
def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    return o

summary = dict(roll_number=ROLL_NUMBER, seed=SEED, split_random_state=int(split_state),
               config=dict(CFG_A=CFG_A, LR_GRID=LR_GRID, N_SEEDS=N_SEEDS, CFG_B=CFG_B, NOISE_RATE=NOISE_RATE),
               selected_lr=SELECTED_LR, evidence_validation=E, evidence_test=E_TEST, q1b=B,
               test_q1a={k: v.tolist() for k, v in TEST_A.items()},
               test_q1b=FINAL_TEST_B.to_dict("records"))
with open(os.path.join(OUT_DIR, "results_summary.json"), "w") as f:
    json.dump(jsonable(summary), f, indent=2)
print("Saved to", OUT_DIR, ":", sorted(f for f in os.listdir(OUT_DIR) if not f.startswith("runs")))

In [ ]:
# ---- Record the software/hardware environment used for this run (for the report and the repository) ----
import sys, platform, subprocess, sklearn, scipy, matplotlib
ENV_USED = dict(python=sys.version.split()[0], platform=platform.platform(), torch=torch.__version__,
                torchvision=torchvision.__version__, numpy=np.__version__, pandas=pd.__version__,
                scikit_learn=sklearn.__version__, scipy=scipy.__version__, matplotlib=matplotlib.__version__,
                cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version() if torch.cuda.is_available() else None,
                device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", amp=USE_AMP)
with open(os.path.join(OUT_DIR, "environment_used.json"), "w") as f:
    json.dump(ENV_USED, f, indent=2)
with open(os.path.join(OUT_DIR, "pip_freeze.txt"), "w") as f:
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
print(ENV_USED)